# Experiment 1: Frozen SapBERT Baseline — No Translation

**Goal:** Evaluate frozen SapBERT (`cambridgeltl/SapBERT-from-PubMedBERT-fulltext`) on cross-lingual biomedical entity linking against an ICD-11 concept index.  
**Condition:** Traditional Chinese surface forms are used **as-is** — no OpenCC conversion, no translation.  
**Corpora:** `english_ncbi`, `simp_chinese`, `trad_chinese`  
**Model:** Completely frozen throughout — `model.eval()` + `torch.no_grad()` everywhere, no optimizer, no gradient updates.

## GPU Check

In [7]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Please enable a GPU runtime in Colab: "
        "Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )

print(f"GPU available: {torch.cuda.get_device_name(0)}")
print(f"CUDA version:  {torch.version.cuda}")

GPU available: Tesla T4
CUDA version:  12.8


## Step 1 — Setup

Install dependencies and configure paths.

In [8]:
!pip install -q transformers faiss-cpu

In [15]:
# Mount Google Drive so we can read the corpus file.
# After running this cell, authorise access in the popup.
from google.colab import drive
drive.mount('/content/drive')

# Verify the mount succeeded
import os
if 'rsem_script' in os.listdir('/content/drive/MyDrive'):
    print("Google Drive connected")
else:
    print("Google Drive not connected")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected


In [9]:
import sys
import transformers
import faiss
import numpy as np
import pandas as pd

print(f"Python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"faiss:        {faiss.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")

Python:       3.12.13
torch:        2.10.0+cu128
transformers: 5.0.0
faiss:        1.13.2
numpy:        2.0.2
pandas:       2.2.2


In [16]:
import os

# -----------------------------------------------------------------------
# Set REPO_ROOT to the folder in your Google Drive that contains:
#  
# -----------------------------------------------------------------------
REPO_ROOT = "/content/drive/MyDrive/2026_Medical_Entity_Linking_project/"  # <-- adjust if needed

DATA_PATH = os.path.join(
    REPO_ROOT,
    "data/no_translation/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl"
)

OUTPUT_DIR    = os.path.join(REPO_ROOT, "experiments/frozen_sapbert/v1-unenriched_index")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "baseline_results_exp1_no_translation.jsonl")
SUMMARY_CSV   = os.path.join(OUTPUT_DIR, "baseline_summary_exp1_no_translation.csv")

SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE    = 32
DEVICE        = torch.device("cuda")

# Sanity-check that the data file exists before going further
assert os.path.exists(DATA_PATH), (
    f"Data file not found: {DATA_PATH}\n"
    "Check that REPO_ROOT is set correctly and the file exists in Google Drive."
)

print(f"Data path:  {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device:     {DEVICE}")

Data path:  /content/drive/MyDrive/2026_Medical_Entity_Linking_project/data/no_translation/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl
Output dir: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v1-unenriched_index
Device:     cuda


## Step 2 — Load and Filter Data

In [18]:
import json
from collections import defaultdict

evaluable = []   # entities with a non-null ontology_id
skipped   = []   # entities with ontology_id == null

with open(DATA_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        corpus = record["source_corpus"]
        for ent in record.get("entities", []):
            row = {
                "entity_id":      ent["entity_id"],
                "surface_form":   ent["surface_form"],
                "source_corpus":  corpus,
                "ontology_id":    ent.get("ontology_id"),
                "ontology_label": ent.get("ontology_label"),
            }
            if ent.get("ontology_id") is None:
                skipped.append(row)
            else:
                evaluable.append(row)

def corpus_counts(records):
    counts = defaultdict(int)
    for r in records:
        counts[r["source_corpus"]] += 1
    return dict(counts)

print(f"Evaluable entities (non-null ontology_id): {len(evaluable):,}")
print(f"  by corpus: {corpus_counts(evaluable)}")
print()
print(f"Skipped entities (null ontology_id): {len(skipped):,}")
print(f"  by corpus: {corpus_counts(skipped)}")

Evaluable entities (non-null ontology_id): 18,264
  by corpus: {'english_ncbi': 2158, 'simp_chinese': 13425, 'trad_chinese': 2681}

Skipped entities (null ontology_id): 12,136
  by corpus: {'english_ncbi': 797, 'simp_chinese': 5946, 'trad_chinese': 5393}


## Step 3 — Build Concept Index

Collect all unique `(ontology_id, ontology_label)` pairs from evaluable entities.  
Encode each `ontology_label` with frozen SapBERT (mean pooling over last hidden state, then L2 normalise).  
Build a FAISS `IndexFlatIP` (inner product = cosine similarity on unit-norm vectors).

In [19]:
from transformers import AutoTokenizer, AutoModel

print(f"Loading SapBERT from '{SAPBERT_MODEL}' ...")
tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
model     = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE)

model.eval()
for param in model.parameters():
    param.requires_grad = False

print("Model loaded and completely frozen.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading SapBERT from 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext' ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and completely frozen.
Total parameters: 109,482,240


In [20]:
def encode_texts(texts, batch_size=BATCH_SIZE):
    """Return L2-normalised float32 embeddings for a list of strings."""
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=64,
                return_tensors="pt",
            ).to(DEVICE)
            output = model(**encoded)
            # Mean pooling over token positions
            attention_mask   = encoded["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
            token_embeddings = output.last_hidden_state                          # (B, T, H)
            summed    = (token_embeddings * attention_mask).sum(dim=1)           # (B, H)
            counts    = attention_mask.sum(dim=1)                                # (B, 1)
            mean_pooled = summed / counts                                        # (B, H)
            all_embeddings.append(mean_pooled.cpu().float().numpy())
    embeddings = np.concatenate(all_embeddings, axis=0)  # (N, H)
    # L2 normalise
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (embeddings / norms).astype(np.float32)

In [21]:
# Collect unique concepts (first seen label wins for duplicate ontology_ids)
concept_map = {}
for ent in evaluable:
    oid = ent["ontology_id"]
    if oid not in concept_map:
        concept_map[oid] = ent["ontology_label"] or ""

index_to_id    = list(concept_map.keys())              # FAISS int -> ontology_id
concept_labels = [concept_map[oid] for oid in index_to_id]

print(f"Unique concepts in index: {len(concept_labels):,}")
print("Encoding concept labels with frozen SapBERT ...")

concept_embeddings = encode_texts(concept_labels)
print(f"Concept embeddings shape: {concept_embeddings.shape}")

dim   = concept_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(concept_embeddings)
print(f"FAISS IndexFlatIP built with {index.ntotal:,} vectors (dim={dim}).")

Unique concepts in index: 2,931
Encoding concept labels with frozen SapBERT ...
Concept embeddings shape: (2931, 768)
FAISS IndexFlatIP built with 2,931 vectors (dim=768).


## Step 4 — Encode Mentions

Encode each evaluable entity's `surface_form` with the same frozen SapBERT encoder.  
**Experiment 1 condition:** Traditional Chinese surface forms are used **as-is** (no OpenCC, no translation).

In [22]:
print(f"Encoding {len(evaluable):,} mention surface forms with frozen SapBERT ...")

surface_forms      = [ent["surface_form"] for ent in evaluable]
mention_embeddings = encode_texts(surface_forms)

print(f"Mention embeddings shape: {mention_embeddings.shape}")

Encoding 18,264 mention surface forms with frozen SapBERT ...
Mention embeddings shape: (18264, 768)


## Step 5 — Retrieve and Evaluate

Retrieve top-10 candidates per mention from the FAISS index.  
Compute Acc@1, Acc@5, Acc@10 for the full dataset and per source corpus.

In [23]:
K = 10
print(f"Searching FAISS index (top-{K}) for {len(evaluable):,} mentions ...")
_scores, _indices = index.search(mention_embeddings, K)  # (N, K)
print("Search complete.")

Searching FAISS index (top-10) for 18,264 mentions ...
Search complete.


In [24]:
per_entity_results = []

for i, ent in enumerate(evaluable):
    gold_uri  = ent["ontology_id"]
    retrieved = [index_to_id[idx] for idx in _indices[i]]  # list of K URIs

    top1_correct  = retrieved[0] == gold_uri
    top5_correct  = gold_uri in retrieved[:5]
    top10_correct = gold_uri in retrieved[:10]

    per_entity_results.append({
        "entity_id":          ent["entity_id"],
        "surface_form":       ent["surface_form"],
        "source_corpus":      ent["source_corpus"],
        "gold_uri":           gold_uri,
        "top1_predicted_uri": retrieved[0],
        "top1_correct":       top1_correct,
        "top5_correct":       top5_correct,
        "top10_correct":      top10_correct,
        "top5_candidates":    retrieved[:5],
    })

print(f"Evaluation complete for {len(per_entity_results):,} entities.")

Evaluation complete for 18,264 entities.


In [25]:
def compute_acc(records):
    n = len(records)
    if n == 0:
        return {"N": 0, "Acc@1": float("nan"), "Acc@5": float("nan"), "Acc@10": float("nan")}
    return {
        "N":      n,
        "Acc@1":  round(sum(r["top1_correct"]  for r in records) / n, 4),
        "Acc@5":  round(sum(r["top5_correct"]  for r in records) / n, 4),
        "Acc@10": round(sum(r["top10_correct"] for r in records) / n, 4),
    }

rows = {}
for corpus in ["english_ncbi", "simp_chinese", "trad_chinese"]:
    subset = [r for r in per_entity_results if r["source_corpus"] == corpus]
    rows[corpus] = compute_acc(subset)
rows["ALL"] = compute_acc(per_entity_results)

summary_df = pd.DataFrame(rows).T[["N", "Acc@1", "Acc@5", "Acc@10"]]
summary_df.index.name = "corpus"

print("\n=== Experiment 1: Frozen SapBERT — No Translation ===")
print(summary_df.to_string())


=== Experiment 1: Frozen SapBERT — No Translation ===
                    N   Acc@1   Acc@5  Acc@10
corpus                                       
english_ncbi   2158.0  0.2326  0.4217  0.5283
simp_chinese  13425.0  0.2638  0.3846  0.4311
trad_chinese   2681.0  0.1268  0.1716  0.1843
ALL           18264.0  0.2400  0.3577  0.4064


In [26]:
summary_df

,N,Acc@1,Acc@5,Acc@10
corpus,,,,
english_ncbi,2158.0,0.2326,0.4217,0.5283
simp_chinese,13425.0,0.2638,0.3846,0.4311
trad_chinese,2681.0,0.1268,0.1716,0.1843
ALL,18264.0,0.2400,0.3577,0.4064


## Step 6 — Save Results

In [27]:
# Per-entity JSONL
with open(RESULTS_JSONL, "w", encoding="utf-8") as fh:
    for row in per_entity_results:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Per-entity results saved to: {RESULTS_JSONL}")

# Summary CSV
summary_df.to_csv(SUMMARY_CSV)
print(f"Summary table saved to:      {SUMMARY_CSV}")

Per-entity results saved to: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v1-unenriched_index/baseline_results_exp1_no_translation.jsonl
Summary table saved to:      /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v1-unenriched_index/baseline_summary_exp1_no_translation.csv


---
## Step 7 — Evaluate on Test Set

Load the held-out test JSONL, filter to evaluable entities (non-null `ontology_id`), encode mentions with the same frozen SapBERT encoder, and retrieve from the **already-built concept index** (single canonical label per URI — unenriched).  
Report Acc@1, Acc@5, Acc@10 for the full test set and per source corpus.  
Save per-entity results and a summary CSV.

> The concept index was built from the training split only — no test surface forms are included.

In [28]:
TEST_DATA_PATH = os.path.join(
    REPO_ROOT,
    "data/no_translation/combined_disease_corpus_test_with_cuis_icd11_cleaned.jsonl"
)

TEST_RESULTS_JSONL = os.path.join(OUTPUT_DIR, "test_results_exp1_no_translation.jsonl")
TEST_SUMMARY_CSV   = os.path.join(OUTPUT_DIR, "test_summary_exp1_no_translation.csv")

assert os.path.exists(TEST_DATA_PATH), (
    f"Test file not found: {TEST_DATA_PATH}\n"
    "Check that REPO_ROOT is set correctly and the file exists in Google Drive."
)
print(f"Test data path: {TEST_DATA_PATH}")


Test data path: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/data/no_translation/combined_disease_corpus_test_with_cuis_icd11_cleaned.jsonl


In [29]:
test_evaluable = []
test_skipped   = []

with open(TEST_DATA_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        corpus = record["source_corpus"]
        for ent in record.get("entities", []):
            row = {
                "entity_id":      ent["entity_id"],
                "surface_form":   ent["surface_form"],
                "source_corpus":  corpus,
                "ontology_id":    ent.get("ontology_id"),
                "ontology_label": ent.get("ontology_label"),
            }
            if ent.get("ontology_id") is None:
                test_skipped.append(row)
            else:
                test_evaluable.append(row)

print(f"Test evaluable entities: {len(test_evaluable):,}")
print(f"  by corpus: {corpus_counts(test_evaluable)}")
print()
print(f"Test skipped (null ontology_id): {len(test_skipped):,}")
print(f"  by corpus: {corpus_counts(test_skipped)}")


Test evaluable entities: 3,311
  by corpus: {'english_ncbi': 361, 'simp_chinese': 2560, 'trad_chinese': 390}

Test skipped (null ontology_id): 1,710
  by corpus: {'english_ncbi': 195, 'simp_chinese': 900, 'trad_chinese': 615}


In [30]:
print(f"Encoding {len(test_evaluable):,} test mention surface forms with frozen SapBERT ...")

test_surface_forms      = [ent["surface_form"] for ent in test_evaluable]
test_mention_embeddings = encode_texts(test_surface_forms)

print(f"Test mention embeddings shape: {test_mention_embeddings.shape}")


Encoding 3,311 test mention surface forms with frozen SapBERT ...
Test mention embeddings shape: (3311, 768)


In [31]:
print(f"Searching concept index (top-{K}) for {len(test_evaluable):,} test mentions ...")
_test_scores, _test_indices = index.search(test_mention_embeddings, K)  # (N, K)
print("Search complete.")


Searching concept index (top-10) for 3,311 test mentions ...
Search complete.


In [32]:
test_per_entity_results = []

for i, ent in enumerate(test_evaluable):
    gold_uri  = ent["ontology_id"]
    retrieved = [index_to_id[idx] for idx in _test_indices[i]]  # list of K URIs

    top1_correct  = retrieved[0] == gold_uri
    top5_correct  = gold_uri in retrieved[:5]
    top10_correct = gold_uri in retrieved[:10]

    test_per_entity_results.append({
        "entity_id":          ent["entity_id"],
        "surface_form":       ent["surface_form"],
        "source_corpus":      ent["source_corpus"],
        "gold_uri":           gold_uri,
        "top1_predicted_uri": retrieved[0],
        "top1_correct":       top1_correct,
        "top5_correct":       top5_correct,
        "top10_correct":      top10_correct,
        "top5_candidates":    retrieved[:5],
    })

print(f"Test evaluation complete for {len(test_per_entity_results):,} entities.")


Test evaluation complete for 3,311 entities.


In [33]:
test_rows = {}
for corpus in ["english_ncbi", "simp_chinese", "trad_chinese"]:
    subset = [r for r in test_per_entity_results if r["source_corpus"] == corpus]
    test_rows[corpus] = compute_acc(subset)
test_rows["ALL"] = compute_acc(test_per_entity_results)

test_summary_df = pd.DataFrame(test_rows).T[["N", "Acc@1", "Acc@5", "Acc@10"]]
test_summary_df.index.name = "corpus"

print("\n=== Experiment 1 TEST SET: Frozen SapBERT — Unenriched Index, No Translation ===")
print(test_summary_df.to_string())



=== Experiment 1 TEST SET: Frozen SapBERT — Unenriched Index, No Translation ===
                   N   Acc@1   Acc@5  Acc@10
corpus                                      
english_ncbi   361.0  0.1247  0.2825  0.3767
simp_chinese  2560.0  0.2574  0.3699  0.4023
trad_chinese   390.0  0.1564  0.2026  0.2231
ALL           3311.0  0.2310  0.3407  0.3784


In [34]:
test_summary_df


,N,Acc@1,Acc@5,Acc@10
corpus,,,,
english_ncbi,361.0,0.1247,0.2825,0.3767
simp_chinese,2560.0,0.2574,0.3699,0.4023
trad_chinese,390.0,0.1564,0.2026,0.2231
ALL,3311.0,0.2310,0.3407,0.3784


### Train vs Test Accuracy Comparison

In [35]:
comparison = pd.concat(
    [summary_df.add_suffix(" (train)"), test_summary_df.add_suffix(" (test)")],
    axis=1,
)[["N (train)", "Acc@1 (train)", "N (test)", "Acc@1 (test)",
   "Acc@5 (train)", "Acc@5 (test)", "Acc@10 (train)", "Acc@10 (test)"]]
comparison.index.name = "corpus"
print(comparison.to_string())
comparison


              N (train)  Acc@1 (train)  N (test)  Acc@1 (test)  Acc@5 (train)  Acc@5 (test)  Acc@10 (train)  Acc@10 (test)
corpus                                                                                                                    
english_ncbi     2158.0         0.2326     361.0        0.1247         0.4217        0.2825          0.5283         0.3767
simp_chinese    13425.0         0.2638    2560.0        0.2574         0.3846        0.3699          0.4311         0.4023
trad_chinese     2681.0         0.1268     390.0        0.1564         0.1716        0.2026          0.1843         0.2231
ALL             18264.0         0.2400    3311.0        0.2310         0.3577        0.3407          0.4064         0.3784


,N (train),Acc@1 (train),N (test),Acc@1 (test),Acc@5 (train),Acc@5 (test),Acc@10 (train),Acc@10 (test)
corpus,,,,,,,,
english_ncbi,2158.0,0.2326,361.0,0.1247,0.4217,0.2825,0.5283,0.3767
simp_chinese,13425.0,0.2638,2560.0,0.2574,0.3846,0.3699,0.4311,0.4023
trad_chinese,2681.0,0.1268,390.0,0.1564,0.1716,0.2026,0.1843,0.2231
ALL,18264.0,0.2400,3311.0,0.2310,0.3577,0.3407,0.4064,0.3784


In [36]:
# Per-entity JSONL
with open(TEST_RESULTS_JSONL, "w", encoding="utf-8") as fh:
    for row in test_per_entity_results:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Test per-entity results saved to: {TEST_RESULTS_JSONL}")

# Summary CSV
test_summary_df.to_csv(TEST_SUMMARY_CSV)
print(f"Test summary table saved to:      {TEST_SUMMARY_CSV}")


Test per-entity results saved to: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v1-unenriched_index/test_results_exp1_no_translation.jsonl
Test summary table saved to:      /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v1-unenriched_index/test_summary_exp1_no_translation.csv


---
## Next: Experiment 2

**Experiment 2** will repeat this exact pipeline with one change: Traditional Chinese surface forms will be converted to Simplified Chinese using **OpenCC** (`t2s` conversion) before encoding with SapBERT. This tests whether script normalisation improves linking accuracy for the `trad_chinese` corpus while leaving `english_ncbi` and `simp_chinese` results unchanged.